<a href="https://colab.research.google.com/github/anirudhjbabu1/LLM/blob/main/MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Start completely fresh
!rm -rf maxtext
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# 2. Install fast package manager
!pip install uv

# 3. Upgrade Core Libraries
!pip install --upgrade numpy jax jaxlib

# 4. Install MaxText directly from your local cloned repository
!uv pip install -e .

# 5. Install the required cutting-edge extensions
!uv pip install qwix tokamax drjax nest_asyncio torch ipywidgets pathwaysutils
!pip install ml_collections tiktoken sentencepiece tensorflow omegaconf

# 6. Force-upgrade Transformers from source for OmniMoE support
!pip install --upgrade --force-reinstall git+https://github.com/huggingface/transformers.git

print("\n✅ Accelerator environment and all dependencies successfully deployed!")

Cloning into 'maxtext'...
remote: Enumerating objects: 97382, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 97382 (delta 57), reused 61 (delta 39), pack-reused 97267 (from 2)
Receiving objects: 100% (97382/97382), 410.56 MiB | 19.28 MiB/s, done.
Resolving deltas: 100% (71473/71473), done.
/content/maxtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 8.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
  Attempt

Using Python 3.12.13 environment at: /usr
Resolved 1 package in 869ms
Prepared 1 package in 1ms
Installed 1 package in 2ms
 + maxtext==0.2.3 (from file:///content/maxtext)
Using Python 3.12.13 environment at: /usr
Resolved 135 packages in 1.10s
Prepared 12 packages in 1.00s
Uninstalled 3 packages in 19ms
Installed 12 packages in 160ms
 - absl-py==1.4.0
 + absl-py==2.4.0
 + drjax==0.2.0
 + einshape==1.0
 - flax==0.11.2
 + flax==0.12.7
 + jaxtyping==0.3.11
 + jedi==0.20.0
 + pathwaysutils==0.1.9
 + qwix==0.1.7
 + tensorboardx==2.6.5
 + tokamax==0.0.12
 - typeguard==4.5.2
 + typeguard==2.13.3
 + wadler-lindig==0.1.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 3.8 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-ak883igt
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-ak883igt
  Resolved https://github.com/huggingface/transformers.git to commit ed51a0798ddf


✅ Accelerator environment and all dependencies successfully deployed!


In [ ]:
!pip install aqtp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.6/901.6 kB 13.5 MB/s eta 0:00:00


In [ ]:
!pip install ml_goodput_measurement

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.8/82.8 kB 4.0 MB/s eta 0:00:00


In [ ]:
import os

# Spoof the FULL XPK distributed environment for a single Colab CPU
os.environ["JOB_INDEX"] = "0"
os.environ["JOB_COMPLETION_INDEX"] = "0"
os.environ["PROCESSES_IN_JOB"] = "1"
os.environ["JAX_PROCESS_COUNT"] = "1"  # 👈 The final missing variable!

In [ ]:
import os
import sys
import nest_asyncio
from absl import logging as absl_logging

absl_logging.set_verbosity(absl_logging.INFO)
nest_asyncio.apply()

# 1. Spoof the XPK distributed environment BEFORE Jax loads
os.environ["JOB_INDEX"] = "0"
os.environ["JOB_COMPLETION_INDEX"] = "0"
os.environ["PROCESSES_IN_JOB"] = "1"
os.environ["JAX_PROCESS_COUNT"] = "1"
os.environ["ENABLE_TE_JAX"] = "0"
os.environ["MAXTEXT_DISABLE_TE"] = "1"

# 1.5. Intercept and bypass JAX's strict initialization locks
import jax
original_init = jax.distributed.initialize
def safe_init(*args, **kwargs):
    try:
        original_init(*args, **kwargs)
    except RuntimeError as e:
        print(f"⚠️ Bypassing JAX distributed init error (safe for single-node CPU): {e}")
jax.distributed.initialize = safe_init

# 2. Map Paths
if os.path.exists("/content/maxtext"):
    os.chdir("/content/maxtext")

for mod in list(sys.modules.keys()):
    if any(pkg in mod for pkg in ["maxtext", "transformers", "drjax"]):
        del sys.modules[mod]

LOCAL_SRC = "/content/maxtext/src"
if LOCAL_SRC in sys.path:
    sys.path.remove(LOCAL_SRC)
sys.path.insert(0, LOCAL_SRC)

MAXTEXT_CONFIGS_DIR = "/content/maxtext/src/maxtext/configs"

# 3. Import and Run!
from maxtext.trainers.pre_train import train

argv_moe = [
    "",
    f"{MAXTEXT_CONFIGS_DIR}/base.yml",
    "model_name=deepseek2-16b",
    "override_model_config=true",
    "base_emb_dim=1024",
    "base_num_decoder_layers=8",
    "base_num_query_heads=8",
    "base_num_kv_heads=8",              # 👈 Fix: Matched Query and KV heads for MLA
    "num_experts=8",
    "num_experts_per_tok=2",
    "base_moe_mlp_dim=2048",
    "steps=50",
    "dataset_type=synthetic",
    "enable_checkpointing=false",
    "base_output_directory=/tmp/maxtext_deepseek",
    "run_name=deepseek_moe_profile",
    "per_device_batch_size=1",
    "hardware=cpu",
    "attention=dot_product",
    "max_target_length=512"
]

print("🚀 Executing Task 3 (DeepSeek MoE) on CPU...")
train.main(argv_moe)

[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as att

[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] vertex_tensorboard: dependency missing; using stub. (ModuleNotFoundError)
🚀 Executing Task 3 (DeepSeek MoE) on CPU...


INFO:absl:Attempting to initialize the jax distributed system for CPU backend...
INFO:absl:Coordinator IP address: 
INFO:absl: Jax process id is 0 
INFO:absl:Jax distributed system initialized on CPUs!
INFO:absl: Setting num_slices=1 for CPU hardware type
INFO:maxtext.configs.pyconfig:Config param abort_on_inf_loss: True
INFO:maxtext.configs.pyconfig:Config param abort_on_nan_loss: True
INFO:maxtext.configs.pyconfig:Config param act_quantization_calibration_method: absmax
INFO:maxtext.configs.pyconfig:Config param activation_dropout_for_audio: 0.0
INFO:maxtext.configs.pyconfig:Config param activation_function_for_audio: gelu
INFO:maxtext.configs.pyconfig:Config param activations_in_float32: False
INFO:maxtext.configs.pyconfig:Config param adam_b1: 0.9
INFO:maxtext.configs.pyconfig:Config param adam_b2: 0.95
INFO:maxtext.configs.pyconfig:Config param adam_eps: 1e-08
INFO:maxtext.configs.pyconfig:Config param adam_eps_root: 0.0
INFO:maxtext.configs.pyconfig:Config param adam_weight_decay

⚠️ Bypassing JAX distributed init error (safe for single-node CPU): maximum recursion depth exceeded⚠️ Bypassing JAX distributed init error (safe for single-node CPU): maximum recursion depth exceeded


ValueError: Only interpret mode is supported on CPU backend.